# 01 — Data Collection & Cleaning

This notebook documents the collection and preprocessing workflow used for the **SHEIN Consumer Reputation Analysis** project.

The project combines three sources:

- **Google News RSS** — SHEIN-related media coverage, 2019–2025
- **Google Trends** — monthly relative search interest, 2019–2025
- **Google Play reviews** — consumer ratings and review text, 2019–Aug. 2024

The final analysis used **3,584 cleaned news headlines**, **84 monthly Google Trends observations**, and **39,797 cleaned English-language Google Play reviews**.

> Google News RSS is a live source, so historical results can change. The original project froze the cleaned dataset after collection so later analyses used the same observations.


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import quote
import re
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

NEWS_RAW_PATH = RAW_DIR / "shein_google_news_raw.csv"
NEWS_CLEAN_PATH = PROCESSED_DIR / "shein_google_news_clean.csv"
TRENDS_PATH = RAW_DIR / "shein_google_trends_monthly.csv"
REVIEWS_PATH = RAW_DIR / "shein_google_play_reviews.csv"


## 1. Google News RSS collection

The original project queried Google News RSS in **weekly intervals** rather than requesting the full seven-year period at once. This improved historical coverage. For each result, the workflow retained headline title, publication date, source, and article URL.

The live collection block is disabled by default because rerunning it today may produce a different historical set than the frozen project dataset.


In [ ]:
RUN_LIVE_COLLECTION = False

def weekly_windows(start_date, end_date):
    current = start_date
    while current <= end_date:
        window_end = min(current + timedelta(days=6), end_date)
        yield current, window_end
        current = window_end + timedelta(days=1)

def collect_google_news_week(start_date, end_date):
    query = (
        f'SHEIN after:{start_date.strftime("%Y-%m-%d")} '
        f'before:{(end_date + timedelta(days=1)).strftime("%Y-%m-%d")}'
    )
    url = "https://news.google.com/rss/search?" + f"q={quote(query)}&hl=en-US&gl=US&ceid=US:en"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    root = ET.fromstring(response.content)

    rows = []
    for item in root.findall(".//item"):
        source = item.find("source")
        rows.append({
            "title": item.findtext("title"),
            "published": item.findtext("pubDate"),
            "source": source.text if source is not None else None,
            "url": item.findtext("link"),
        })
    return rows

if RUN_LIVE_COLLECTION:
    all_rows = []
    for start, end in weekly_windows(datetime(2019,1,1), datetime(2025,12,31)):
        all_rows.extend(collect_google_news_week(start, end))
        time.sleep(0.5)
    news_raw = pd.DataFrame(all_rows)
    news_raw.to_csv(NEWS_RAW_PATH, index=False)
    print(f"Saved {len(news_raw):,} raw RSS results")
else:
    print("Live collection disabled. Use the frozen project dataset when available.")


## 2. Clean the news dataset

Cleaning included standardized dates, missing-text removal, deduplication, obvious irrelevant-result filtering, source-suffix cleanup, complete-word matching for `SHEIN`, and manual quality checks for false positives. The original cleaned dataset contained **3,584 unique headlines**.


In [ ]:
def clean_news(news):
    df = news.copy()
    df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True).dt.tz_convert(None)
    df = df.dropna(subset=["title", "published"]).copy()
    df["title"] = df["title"].astype(str).str.strip()
    df = df.drop_duplicates(subset=["title"])

    if "source" in df.columns:
        df = df[~df["source"].fillna("").str.contains("youtube", case=False, regex=False)]

    def strip_source_suffix(row):
        title = row["title"]
        source = str(row.get("source", "") or "").strip()
        suffix = f" - {source}"
        return title[:-len(suffix)].strip() if source and title.endswith(suffix) else title

    df["title"] = df.apply(strip_source_suffix, axis=1)
    df = df[df["title"].str.contains(r"\bSHEIN\b", case=False, regex=True, na=False)].copy()
    df = df[df["published"].between(pd.Timestamp("2019-01-01"), pd.Timestamp("2025-12-31 23:59:59"))].copy()
    df["month"] = df["published"].dt.to_period("M").astype(str)
    df["year"] = df["published"].dt.year
    return df.reset_index(drop=True)

# news_raw = pd.read_csv(NEWS_RAW_PATH)
# news_clean = clean_news(news_raw)
# news_clean.to_csv(NEWS_CLEAN_PATH, index=False)


## 3. Ethics-related media classification

The project used a manually constructed dictionary spanning four non-mutually-exclusive categories: **labor and working conditions**, **environment and sustainability**, **intellectual property**, and **supply chain and regulation**. Whole-word / whole-phrase matching reduced false positives.


In [ ]:
ETHICS_KEYWORDS = {
    "labor": ["worker", "workers", "wage", "wages", "forced labor", "working conditions", "labor rights", "labour rights"],
    "environment": ["sustainability", "sustainable", "pollution", "emissions", "waste", "greenwashing", "environment", "environmental"],
    "intellectual_property": ["copyright", "plagiarism", "plagiarized", "counterfeit", "trademark", "intellectual property"],
    "supply_chain_regulation": ["supply chain transparency", "supply chain", "investigation", "regulator", "regulation", "sanctions", "customs"],
}

def contains_phrase(text, phrase):
    pattern = rf"(?<!\w){re.escape(phrase)}(?!\w)"
    return bool(re.search(pattern, str(text), flags=re.IGNORECASE))

def classify_ethics_topics(title):
    matched = []
    for category, phrases in ETHICS_KEYWORDS.items():
        if any(contains_phrase(title, phrase) for phrase in phrases):
            matched.append(category)
    return matched

def add_ethics_classification(news_clean):
    df = news_clean.copy()
    df["ethics_topics"] = df["title"].apply(classify_ethics_topics)
    df["ethics_related"] = df["ethics_topics"].str.len().gt(0)
    for category in ETHICS_KEYWORDS:
        df[f"ethics_{category}"] = df["ethics_topics"].apply(lambda topics: category in topics)
    return df


## 4. Monthly media metrics

Two core measures feed the integrated analysis: **ethics-related media volume** and **ethics-related media share**.


In [ ]:
def build_monthly_media(news):
    monthly = (news.groupby("month").agg(
        total_headlines=("title", "size"),
        ethics_related_volume=("ethics_related", "sum")
    ).reset_index())
    monthly["ethics_related_share"] = 100 * monthly["ethics_related_volume"] / monthly["total_headlines"]
    return monthly


## 5. Google Trends

The project used **84 monthly observations** for the term `SHEIN` from Jan. 2019 through Dec. 2025. Google Trends is a normalized index from 0 to 100, not raw search volume.


In [ ]:
def load_google_trends(path=TRENDS_PATH):
    trends = pd.read_csv(path)
    trends["month"] = pd.to_datetime(trends["month"], errors="coerce").dt.to_period("M").astype(str)
    trends["search_interest"] = pd.to_numeric(trends["search_interest"], errors="coerce")
    return trends.dropna(subset=["month", "search_interest"]).drop_duplicates(subset=["month"]).sort_values("month").reset_index(drop=True)


## 6. Google Play review preparation

Historical reviews came from the **ShoppingAppReviews Dataset (Version 2)** on Mendeley Data. The project retained SHEIN reviews, standardized timestamps, removed records without written content, restricted the study period, and produced **39,797 cleaned English-language reviews**.


In [ ]:
def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise KeyError(f"None of these columns were found: {candidates}")

def clean_reviews(path=REVIEWS_PATH):
    reviews = pd.read_csv(path)
    app_col = next((c for c in ["app", "app_name", "application", "App"] if c in reviews.columns), None)
    text_col = first_existing(reviews.columns, ["review", "content", "review_text", "text"])
    rating_col = first_existing(reviews.columns, ["rating", "score", "stars", "star_rating"])
    date_col = first_existing(reviews.columns, ["date", "timestamp", "at", "review_date"])

    if app_col:
        reviews = reviews[reviews[app_col].astype(str).str.contains("shein", case=False, na=False)].copy()

    reviews["review_text"] = reviews[text_col].astype("string").str.strip()
    reviews["rating"] = pd.to_numeric(reviews[rating_col], errors="coerce")
    reviews["review_date"] = pd.to_datetime(reviews[date_col], errors="coerce")
    reviews = reviews.dropna(subset=["review_text", "rating", "review_date"]).copy()
    reviews = reviews[reviews["review_text"].str.len().gt(0)]
    reviews = reviews[reviews["review_date"].between(pd.Timestamp("2019-01-01"), pd.Timestamp("2024-08-31 23:59:59"))].copy()
    reviews["month"] = reviews["review_date"].dt.to_period("M").astype(str)
    return reviews.reset_index(drop=True)


## Output

The cleaned and monthly datasets feed Notebook 02, which performs sentiment analysis, multi-source integration, correlation testing, one-month lag analysis, period comparisons, and business interpretation. Correlations are treated as **associations rather than causal effects**.
